In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(r"D:\AI Engineering\RAG\iso27001.pdf")
documents = loader.load()

print(len(documents))
print(documents[0])

d:\AI Engineering\RAG\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
d:\AI Engineering\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


26
page_content='Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèmes de management de la sécurité de l'information — 
Exigences
INTERNATIONAL 
STANDARD
ISO/IEC 
27001
Third edition  
2022-10
Reference number 
ISO/IEC 27001:2022(E)
© ISO/IEC 2022
--``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`---' metadata={'producer': 'PDPreStamp v3.3', 'creator': 'Adobe InDesign 16.4 (Windows)', 'creationdate': '2022-09-29T17:54:00+02:00', 'author': 'ISO', 'license': 'Information Handling Services, 2022', 'moddate': '2022-10-26T19:22:40+08:00', 'title': 'ISO/IEC 27001:2022', 'trapped': '/False', 'source': 'D:\\AI Engineering\\RAG\\iso27001.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)
docs = text_splitter.split_documents(documents)

# Now 'docs' can be used for embeddings
print(docs[0].page_content)

Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèmes de management de la sécurité de l'information — 
Exigences
INTERNATIONAL 
STANDARD
ISO/IEC 
27001
Third edition  
2022-10
Reference number 
ISO/IEC 27001:2022(E)
© ISO/IEC 2022
--``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`---


### RAG Pipeline Completion

To finish our simple RAG system, we need to do three main things:
1. Initialize an **Embedding Model** and store our chunks into a **Vector Database** (`Chroma`).
2. Initialize a **Large Language Model (LLM)** and set up our Retrieval-Augmented Generation (RAG) chain.
3. Invoke the chain with a query.

In [12]:
# If you missing these packages, uncomment and run this cell:
# !pip install sentence-transformers chromadb langchain_openai

In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Initialize an open-source embedding model (downloads automatically)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. Store the chunks into the Chroma vector database
vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings)

# 3. Create a retriever to retrieve the most relevant chunks
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Vector store created and retriever initialized!")

C:\Users\LAXMAN\AppData\Local\Temp\ipykernel_5952\2705039709.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3972.90it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store created and retriever initialized!


In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# Load environment variables from .env file
load_dotenv()

# The ChatGoogleGenerativeAI automatically reads the GOOGLE_API_KEY from the loaded environment variables
# Using gemini-2.5-flash since gemini-1.5-flash is not available for this API version
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Define the system prompt for our RAG application
system_prompt = (
    "You are an expert assistant. Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you don't know.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Create the RAG chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("RAG chain is ready with Gemini!")

RAG chain is ready with Gemini!


In [ ]:
# Let's query our RAG system!
query = "What is an information security management system according to the document?"
response = rag_chain.invoke({"input": query})
print(response["answer"])

According to the document, an information security management system (ISMS) is a system that an organization establishes, implements, maintains, and continually improves.

Its primary purpose is to:
*   **Preserve the confidentiality, integrity, and availability of information** by applying a risk management process.
*   **Give confidence to interested parties that risks are adequately managed.**

The adoption of an ISMS is considered a strategic decision for an organization. It is expected to be part of and integrated with the organization’s processes and overall management structure, ensuring information security is considered in the design of processes, information systems, and controls. It includes the processes needed and their interactions.
